# Shannon Entropy — Interactive Notebook

**Concept 36** of the math-foundations learning map.

Shannon entropy of a discrete RV $X$ with pmf $p$:
$$H(X) = \mathbb{E}[-\log p(X)] = -\sum_x p(x) \log p(x).$$

We use $\log_2$ (units = bits) throughout. **Stdlib only** — no numpy/scipy.

Cells:
1. Imports and helpers
2. Shannon entropy implementation
3. Bernoulli entropy curve (text plot)
4. Bound $H \le \log_2 n$ on $n=4$ outcomes
5. Akgül §3.1 connection: softmax temperature vs entropy
6. Subadditivity sanity check
7. Summary


In [ ]:
from math import log2, exp

def H(probs):
    """Shannon entropy in bits."""
    return -sum(p * log2(p) for p in probs if p > 0.0)

def softmax(logits, T):
    scaled = [z / T for z in logits]
    m = max(scaled)
    exps = [exp(z - m) for z in scaled]
    Z = sum(exps)
    return [e / Z for e in exps]

def bar(value, vmax, width=40):
    n = int(round(width * value / vmax)) if vmax > 0 else 0
    return '#' * n

print('helpers loaded')


## Cell 3 — Bernoulli entropy curve

$H(p) = -p \log_2 p - (1-p)\log_2(1-p)$ peaks at $p = 1/2$ with value $1$ bit.


In [ ]:
ps = [0.05 * k for k in range(1, 20)]
Hs = [H([p, 1 - p]) for p in ps]
Hmax = max(Hs)
for p, h in zip(ps, Hs):
    print(f'p={p:0.2f}  H={h:0.4f}  |{bar(h, Hmax)}')
pstar = ps[Hs.index(Hmax)]
print(f'\nargmax at p={pstar:0.2f} with H={Hmax:0.4f} bits (theory: p=0.5, H=1)')


## Cell 4 — Maximum entropy bound on $n=4$ outcomes

Theorem: $H(X) \le \log_2 n$, with equality iff $X$ is uniform.


In [ ]:
log2n = log2(4)
dists = {
    'uniform        ': [0.25, 0.25, 0.25, 0.25],
    'mildly skewed  ': [0.40, 0.30, 0.20, 0.10],
    'heavily skewed ': [0.85, 0.10, 0.04, 0.01],
    'deterministic  ': [1.00, 0.00, 0.00, 0.00],
}
for name, p in dists.items():
    h = H(p)
    ok = 'OK' if h <= log2n + 1e-12 else 'FAIL'
    print(f'{name} H={h:0.4f}  <=  log2(4)={log2n:0.4f}  [{ok}]')


## Cell 5 — Akgül §3.1 Eq. 1: per-token entropy

Per-token entropy $H_t = -\sum_v p_\theta(v\mid x_{<t}) \log p_\theta(v\mid x_{<t})$
is exactly Shannon entropy of the next-token softmax. We sweep temperature $T$:
low $T$ collapses mass onto the argmax (low entropy), high $T$ flattens to uniform (max entropy).


In [ ]:
logits = [3.0, 1.5, 0.5, 0.0, -1.0]
print(f'logits = {logits}')
print(f'log2(5) = {log2(5):0.4f}  (uniform upper bound)\n')
for T in [0.25, 0.5, 1.0, 2.0, 4.0, 16.0, 64.0]:
    p = softmax(logits, T)
    h = H(p)
    probs_str = '[' + ', '.join(f'{q:0.3f}' for q in p) + ']'
    print(f'T={T:6.2f}  H={h:0.4f}  p={probs_str}')


## Cell 6 — Subadditivity sanity check

$H(X, Y) \le H(X) + H(Y)$, with equality iff $X \perp Y$.
We compare the independent product $p(x)q(y)$ against a correlated joint.


In [ ]:
px = [0.5, 0.5]
py = [0.7, 0.3]
indep = [px[i] * py[j] for i in range(2) for j in range(2)]
# Correlated joint: same marginals as indep but with positive correlation.
corr = [0.45, 0.05, 0.25, 0.25]  # marginals: x=(0.5,0.5), y=(0.7,0.3)
Hx = H(px); Hy = H(py)
print(f'H(X)={Hx:0.4f}  H(Y)={Hy:0.4f}  H(X)+H(Y)={Hx+Hy:0.4f}')
print(f'H(X,Y) independent = {H(indep):0.4f}  (should equal H(X)+H(Y))')
print(f'H(X,Y) correlated  = {H(corr):0.4f}  (should be strictly less)')


## Cell 7 — Summary

- Shannon entropy averages self-information; units depend on $\log$ base (bits / nats / hartleys).
- Bernoulli entropy peaks at $p = 1/2$; this is a 1-d instance of the maximum-entropy theorem.
- Uniform distribution saturates $H \le \log_2 n$.
- Per-token softmax entropy (Akgül Eq. 1) is exactly this object; temperature $T$ smoothly
  interpolates between argmax-confidence ($H \to 0$) and uniform-uncertainty ($H \to \log_2 n$).
- Independence saturates subadditivity; correlation strictly reduces joint entropy.

Next: [37-cross-entropy](../37-cross-entropy/README.md) builds on this with $H(p, q)$ and KL divergence.
